In [20]:
# Single sparse columns
single_sparse_col = [
	"brand_id",
	"category_id",
	"root_category_id",
	"polygon_id"
]

# Multi sparse columns (list of lists for multi‑hot encoding)
multi_sparse_col = [
	[]
]

# All dense features (both user and item sides, plus order_hour)
dense_col = [
	# Item dense features
	'avg_availability', 'std_availability',
	'avg_price', 'std_price',
	'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price',
	'total_popularity', 'recent_popularity', 'popularity_trend',
	'item_discount_ratio', 'item_avg_ordering_hour', 'num_products', 'num_sellers',
	'item_age_days',
	'category_availability', 'category_price', 'category_popularity',
	'brand_availability', 'brand_price', 'brand_popularity',
	'brand_category_availability_score', 'brand_category_price_score',
	'brand_category_popularity_score',

	# User dense features
	"recency", "length", "frequency", "monetary",
	"brand_diversity", "category_diversity", "root_category_diversity",
	"user_avg_ordering_hour", "ipo", "lpo", "aov",
	"avg_item_spending", "min_item_spending", "max_item_spending",
	"p25_item_spending", "p50_item_spending", "p75_item_spending",
	"user_discount_ratio",

	# Interaction‑level feature
	"order_hour",
	# "time"
]

# Features belonging to the user side
user_col = [
	# User sparse
	"polygon_id",

	# User dense
	"recency", "length", "frequency", "monetary",
	"brand_diversity", "category_diversity", "root_category_diversity",
	"user_avg_ordering_hour", "ipo", "lpo", "aov",
	"avg_item_spending", "min_item_spending", "max_item_spending",
	"p25_item_spending", "p50_item_spending", "p75_item_spending",
	"user_discount_ratio",

	# Interaction‑level (treated as user feature for simplicity)
	"order_hour",

	# User multi‑sparse
	# "category_id1", "category_id2", "category_id3", "category_id4", "category_id5",
	# "category_id6", "category_id7", "category_id8", "category_id9", "category_id10",
	# "brand_id1", "brand_id2", "brand_id3", "brand_id4", "brand_id5",
	# "brand_id6", "brand_id7", "brand_id8", "brand_id9", "brand_id10"
]

# Features belonging to the item side
item_col = [
	# Item sparse
	"brand_id",
	"category_id",
	"root_category_id",

	# Item dense
	'avg_availability', 'std_availability',
	'avg_price', 'std_price',
	'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price',
	'total_popularity', 'recent_popularity', 'popularity_trend',
	'item_discount_ratio', 'item_avg_ordering_hour', 'num_products', 'num_sellers',
	'item_age_days',
	'category_availability',
	'category_price', 
	'category_popularity',
	'brand_availability', 'brand_price', 'brand_popularity',
	'brand_category_availability_score', 'brand_category_price_score',
	'brand_category_popularity_score'
]

In [ ]:
import pandas as pd
from recora.data import split_by_ratio_chrono

# ── 1. Load raw CSVs ──────────────────────────────────────────────────────────
item_df = pd.read_csv(
    './sample_data/librec_two_tower_category_brand_items_df_v2.csv',
    index_col=0
).rename(columns={'brand_category_id': 'item'})

user_df = pd.read_csv(
    './sample_data/librec_two_tower_category_brand_user_df_v2.csv',
    index_col=0
).rename(columns={'user_id': 'user'})

inter_events = pd.read_csv(
    './sample_data/librec_two_tower_category_brand_inter_events_v2.csv',
    index_col=0
).rename(columns={
    'brand_category_id': 'item',
    'user_id': 'user',
    'order_date': 'time'
})

filtered_users = inter_events.groupby('user')['item'].count().reset_index()
filtered_users = filtered_users[filtered_users['item'] > 10]

filtered_items = inter_events.groupby('item')['user'].count().reset_index()
filtered_items = filtered_items[filtered_items['user'] > 100]

inter_events = inter_events.loc[
    (inter_events['user'].isin(filtered_users['user'])) &
    (inter_events['item'].isin(filtered_items['item']))
]

inter_events['user'] = inter_events['user'].astype(str)
inter_events['item'] = inter_events['item'].astype(str)

item_df['item'] = item_df['item'].astype(str)
user_df['user'] = user_df['user'].astype(str) 

# ── 2. Merge interactions with user/item features ─────────────────────────────
df = inter_events \
    .merge(user_df, on='user', how='inner') \
    .merge(item_df, on='item', how='inner')

# ── 3. Item popularity (for popular negative sampling) ────────────────────────
item_popularity = df['item'].value_counts().to_dict()

# ── 4. Train/eval split (chronological) ──────────────────────────────────────
# train_data, eval_data = split_by_ratio_chrono(df, order=True, test_size=0.2)

# 4. LightGBM-specific: encode categoricals
for col in single_sparse_col:
    df[col] = df[col].astype('category')


# ── 5. Add frequency/recency sample weights ───────────────────────────────────
def add_frequency_recency_sample_weight(frame, user_col="user", item_col="item", time_col="time"):
    weighted = frame.copy()
    # frequency: how many times user interacted with item
    freq = weighted.groupby([user_col, item_col])[time_col].transform('count')
    # recency: rank by time per user (more recent = higher rank)
    recency = weighted.groupby(user_col)[time_col].rank(method='dense', ascending=True)
    weighted['sample_weight'] = freq * recency
    # normalize to [0, 1]
    weighted['sample_weight'] = weighted['sample_weight'] / weighted['sample_weight'].max()
    return weighted

# df = add_frequency_recency_sample_weight(df)

# ── 6. feature_data: item + user features joined on interactions ──────────────
# All columns except interaction metadata become features for the reranker
feature_cols = [c for c in df.columns if c not in ['time', 'sample_weight']]
df = df[feature_cols]

# Verify
print("df columns:", df.columns.tolist())
print("df shape:", df.shape)
df[["user", "item"]].head()

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split

# Negative sampling configuration
NEG_SAMPLING = True
NEG_SAMPLER = "popular"  # random | unconsumed | popular
NUM_NEG = 1
BATCH_SIZE = 1024

def generate_negative_samples(df, num_neg, strategy="popular", item_popularity=None):
    all_items = df['item'].unique()
    
    if strategy == "popular":
        weights = np.array([item_popularity.get(i, 1) for i in all_items], dtype=np.float32)
        weights = weights / weights.sum()
    
    user_items = df.groupby('user')['item'].apply(set).to_dict()
    
    rows = []
    for user, pos_items in user_items.items():
        pos_list = list(pos_items)
        
        # sample all negatives at once per user
        candidates = np.random.choice(all_items, size=num_neg * len(pos_list) * 2,
                                      replace=True, p=weights if strategy == "popular" else None)
        candidates = [c for c in candidates if c not in pos_items][:num_neg * len(pos_list)]
        
        for item in pos_list:
            rows.append((user, item, 1))
        for item in candidates:
            rows.append((user, item, 0))
    
    return pd.DataFrame(rows, columns=['user', 'item', 'label'])

from tqdm import tqdm

def prepare_ranking_data(features_df, num_neg=5, neg_sampler="popular"):
    """Prepare data for LightGBM ranking with negative sampling - vectorized"""
    
    all_items = features_df['item'].unique()
    feature_cols = [c for c in features_df.columns if c not in ['user', 'item', 'label']]
    
    # ── 1. Build fast feature lookup: (user, item) -> feature vector ──────────
    features_df = features_df.copy()
    feature_index = features_df.set_index(['user', 'item'])[feature_cols]
    
    # ── 2. Popularity weights (computed once) ─────────────────────────────────
    if neg_sampler == "popular":
        weights = np.array([item_popularity.get(i, 1) for i in all_items], dtype=np.float32)
        weights = weights / weights.sum()
    else:
        weights = None
    
    # ── 3. Build positive pairs ───────────────────────────────────────────────
    pos_df = features_df[['user', 'item']].copy()
    pos_df['label'] = 1

    # ── 4. Vectorized negative sampling per user ──────────────────────────────
    user_items_map = features_df.groupby('user')['item'].apply(set).to_dict()
    
    neg_rows = []
    for user, pos_items in tqdm(user_items_map.items(), desc="Neg sampling"):
        n_neg = num_neg * len(pos_items)
        
        # oversample then filter consumed items
        candidates = np.random.choice(all_items, size=n_neg * 3, replace=True, p=weights)
        candidates = [c for c in candidates if c not in pos_items][:n_neg]
        
        neg_rows.extend([(user, item, 0) for item in candidates])
    
    neg_df = pd.DataFrame(neg_rows, columns=['user', 'item', 'label'])
    
    # ── 5. Combine pos + neg ──────────────────────────────────────────────────
    all_pairs = pd.concat([pos_df, neg_df], ignore_index=True)
    
    # ── 6. Single vectorized feature join ─────────────────────────────────────
    # For negatives: user/item pairs may not exist in features_df
    # so we join item features and user features separately
    
    item_features = features_df.drop_duplicates('item').set_index('item')[[
        c for c in feature_cols if c in features_df.columns
    ]]
    user_features = features_df.drop_duplicates('user').set_index('user')[[
        c for c in feature_cols if c in features_df.columns  
    ]]
    
    # safer: merge instead of index lookup
    result = all_pairs \
        .merge(features_df.drop_duplicates(['user','item']), on=['user','item'], how='left')
    
    # fill NaN for negative samples that have no feature row
    result[feature_cols] = result[feature_cols].fillna(0)
    
    # ── 7. Build qid (group id per user, required for lambdarank) ─────────────
    result['qid'] = result.groupby('user').ngroup()
    result = result.sort_values('qid')  # lgb requires sorted qid
    
    X   = result[feature_cols].values
    y   = result['label'].values
    qid = result['qid'].values
    
    return X, y, qid


# ── Run ───────────────────────────────────────────────────────────────────────
X, y, qid = prepare_ranking_data(df, num_neg=NUM_NEG, neg_sampler=NEG_SAMPLER)

print(f"X: {X.shape}, y: {y.shape}, qid: {qid.shape}")
print(f"Positive samples: {y.sum()}, Negative: {(y==0).sum()}")

# Prepare data
X, y, qid = prepare_ranking_data(df.head(100), num_neg=NUM_NEG, neg_sampler=NEG_SAMPLER)

In [ ]:
# Split by query groups (users)
unique_qids = np.unique(qid)
train_qids, val_qids = train_test_split(unique_qids, test_size=0.2, random_state=42)

train_mask = np.isin(qid, train_qids)
val_mask = np.isin(qid, val_qids)

X_train, y_train, qid_train = X[train_mask], y[train_mask], qid[train_mask]
X_val, y_val, qid_val = X[val_mask], y[val_mask], qid[val_mask]

# Create LightGBM datasets for ranking
train_data = lgb.Dataset(X_train, label=y_train, group=np.bincount(qid_train))
val_data = lgb.Dataset(X_val, label=y_val, group=np.bincount(qid_val), reference=train_data)

# LightGBM parameters for LambdaRank (evolution of RankNet)
params = {
    'objective': 'lambdarank',  # Uses LambdaRank (improved RankNet with NDCG optimization)
    'metric': ['ndcg', 'map'],
    'ndcg_eval_at': [5, 10, 20],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbose': 1
}

# Train the reranker
print("Training LightGBM LambdaRank reranker...")
ranker = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=10)
    ]
)

# Prediction and evaluation
def predict_and_rank(model, user_features, top_k=10):
    """Predict scores and return top-k items"""
    scores = model.predict(user_features)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices, scores[top_indices]

# Save model
ranker.save_model('lightgbm_reranker.txt')


,order_hour,user,jet_discount,voucher_discount,item,polygon_id,recency,frequency,monetary,length,...,item_age_days,category_availability,category_price,category_popularity,brand_availability,brand_price,brand_popularity,brand_category_availability_score,brand_category_price_score,brand_category_popularity_score
0,14,36258844,0,269500,19801289,NaN,28.0,2.0,37812000.0,28.0,...,358,0.374878,1.727538e+06,916.987342,0.416494,836158.112113,6848.5,0.713213,0.400932,27.755018
1,17,30499466,0,0,19801289,88.0,13.0,5.0,41377500.0,33.0,...,358,0.374878,1.727538e+06,916.987342,0.416494,836158.112113,6848.5,0.713213,0.400932,27.755018
2,18,29909579,0,115500,19801289,89.0,37.0,5.0,39276231.0,84.0,...,358,0.374878,1.727538e+06,916.987342,0.416494,836158.112113,6848.5,0.713213,0.400932,27.755018
3,14,163907,0,0,19801289,89.0,7.0,22.0,205739118.0,87.0,...,358,0.374878,1.727538e+06,916.987342,0.416494,836158.112113,6848.5,0.713213,0.400932,27.755018
4,12,35666527,0,153950,19801289,NaN,1.0,83.0,398846624.0,70.0,...,358,0.374878,1.727538e+06,916.987342,0.416494,836158.112113,6848.5,0.713213,0.400932,27.755018
